# Module 4: Tool returns that tell the truth

A stock check for a depot that doesn't exist comes back looking like "no stock", and the agent answers with confidence. You find where the tools hide the failure, copy in tools whose results say what went wrong, and check that the agent passes that on.

In [ ]:
# Load the lab's notebook helpers, then work from the agent/ folder.
import labnb as lab
%cd ../agent

## Start from module 4

This puts the agent in the state module 4 starts from. Any edits you made are copied to
`agent/.reset-backup/` first, so nothing you wrote is lost.

In [ ]:
# Put the agent back where module 4 starts. Your own edits are backed up first.
!python reset.py 4

## 1. Ask about a depot that doesn't exist

Kestrel has three depots: Denver, Phoenix and Chicago. Portland isn't one of them. The agent's stock
data lives in [`fixtures/stock.json`](../agent/fixtures/stock.json), a small file that lists the
depots and how many of each part each one holds.

The question below asks about Portland anyway. Keep in mind that the right answer is "Portland isn't
a Kestrel depot".

In [ ]:
# Ask the agent the lab's question 'm4':
#   "Is the K-600 high-pressure sensor available at the Portland depot?"
!python agent.py --q m4
# Show a table of this notebook's runs, one row each:
#   turns: how many times the agent called the model
#   ended by: what stopped the run
#   tools run: the tools the agent ran. Once a tool's result has a category, it shows in
#     parentheses after the tool's name. None do yet.
#   answer: what the agent printed
lab.evidence(columns=("turns", "ended by", "tools run", "answer"))

In the lab's test runs, the agent said the sensor is "not currently available at the Portland
depot". That's wrong. Portland isn't a depot, but the answer reads as if the agent checked Portland's
stock and found none. Nothing in it suggests a problem.

## 2. Read what the tool returned

To see why, look at what the tools gave the agent. Below, each tool call is listed with its
arguments and exactly what it returned, under a link to the run's trace.

In [ ]:
# A link to the newest run's trace in MLflow.
lab.trace_link()
# Show each tool the newest run called, with its arguments and exactly what it returned.
lab.tool_results()

`find_part` found the sensor's part number, `check_stock` checked that part at Portland, and the run
ended cleanly on `stop`. What `check_stock` returned is a valid object, `{"results": []}`: an empty
list, with nothing to say that anything went wrong.

Two functions in [`tools.py`](../agent/tools.py) decide what came back. `tools.py` holds the agent's
tools and the wrapper that runs them.

The wrapper is `run_tool`, and the agent runs every tool through it. It looks up the tool by name and
calls it with the arguments the model sent (`**arguments` passes each one in by name). If that
raises an error, the `except` block decides what comes back. Read which errors it catches.

In [ ]:
# Show run_tool from tools.py: the wrapper every tool call goes through.
lab.show_code("tools.py", "run_tool")

`check_stock` reads `fixtures/stock.json` and keeps the rows for the part and depot it was asked
about. Read what it does when the depot isn't in the list. One piece of Python helps here:
`if depot_name and ...` is only true when `depot_name` has a value, and `None` counts as false.

In [ ]:
# Show check_stock from tools.py: the tool that checked Portland.
lab.show_code("tools.py", "check_stock")
# Print the stock data it reads: the depots, then each part's stock at each depot.
!cat fixtures/stock.json

**Your read:** the first run's `check_stock` returned `{"results": []}`. Going by `run_tool` and
`check_stock` in `agent/tools.py` (the notebook shows both), what else could have produced exactly
that result?

<details><summary>Check your answer</summary>

`run_tool` catches every error and returns `{"results": []}`. A wrong argument, an unknown tool, a
part number that isn't Kestrel's and a data file it can't read all come back looking like an empty
search.

`check_stock` doesn't recognise Portland, so `depot_name` is `None` and the `if depot_name and ...`
line never filters by depot. It returns stock from every depot. The K-600 sensor has no stock at any
depot, so the list comes back empty. Ask about a part that is in stock, such as the K-400 coolant
pump assembly, `KI-CP-40817`, and the result for Portland would hold Denver's and Chicago's stock.

Either way, the model can't tell "nothing in stock" from "that depot doesn't exist".
</details>

## 3. Copy in tools that say what went wrong

The lab supplies new versions of two files in `solutions/module-04/`:
[`tools.py`](../agent/solutions/module-04/tools.py) and [`agent.py`](../agent/solutions/module-04/agent.py),
the agent's loop. Copy both over the agent's own files.

In [ ]:
# Copy the supplied tools.py and agent.py over the agent's own copies.
!cp solutions/module-04/tools.py solutions/module-04/agent.py .
# Confirm both files changed: one line per file, with how many lines differ from the starter code.
!git diff --stat -- tools.py agent.py

The output should list both `tools.py` and `agent.py`. The count for `agent.py` includes module 3's
loop fix as well, because it compares with the starter code.

In the new `tools.py`, every tool returns the same four fields, built by a new function, `outcome`:

- `category`: what happened. It's one of `ok`, `no_results`, `bad_request` or `upstream_unavailable`.
- `retryable`: whether trying again could help.
- `detail`: what went wrong, in words.
- `data`: the results, or `null` when the call failed.

These four fields are the tools' *contract*: every result has them, and the docstring at the top
of the new `tools.py` sets them out. In `outcome`, an argument a tool leaves out takes the value after
its `=`, so `retryable` is `False` and `data` is `None` (JSON's `null`) unless the tool says
otherwise.

`bad_request` means the call itself was wrong, like an unknown depot. `check_stock` now returns it
for Portland, with the real depots named in `detail`. `upstream_unavailable` means the system behind
the tool couldn't be reached or failed. This lab's tools read local files, so you shouldn't see it unless a data
file can't be read.
A real stock service that timed out would produce it.

```mermaid
flowchart TB
  subgraph Before["Before: four outcomes, one reply"]
    B1["nothing matched"] --> E1["results: [] (empty)"]
    B2["unknown depot"] --> E1
    B3["backend unreachable"] --> E1
    B4["bad arguments"] --> E1
  end
  subgraph After["After: the error contract"]
    A1["nothing matched"] --> C1["no_results"]
    A2["unknown depot"] --> C2["bad_request, naming the real depots"]
    A3["backend unreachable"] --> C3["upstream_unavailable, retryable"]
    A4["found it"] --> C4["ok, with the data"]
  end
  Before ~~~ After
```

Compare the new `check_stock` and `run_tool` below with the versions in step 2.

In [ ]:
# Show outcome from the new tools.py: the function every tool now builds its result with.
lab.show_code("tools.py", "outcome")
# Show the new check_stock: it rejects a depot it doesn't know instead of skipping the filter.
lab.show_code("tools.py", "check_stock")
# Show the new run_tool: it sorts errors into categories instead of hiding them.
lab.show_code("tools.py", "run_tool")

The supplied `agent.py` changes in two ways. If a result says `retryable: true`, it runs the tool
once more. If a result is `bad_request` or `upstream_unavailable`, it adds a line to the system prompt
telling the model to explain the error and not describe it as out of stock. A clear tool result
helps, but the model still needs telling what to do with it.

To see only module 4's change to `agent.py`, compare the new file with module 3's version, which the
reset gave you. Lines starting with `+` are the ones module 4 adds. The comment above the first added
lines is a note for the lab's maintainers about later modules, so you can skip it.

In [ ]:
# Compare module 3's agent.py with the one you just copied in, and print the lines that differ.
!diff -u solutions/module-03/agent.py agent.py

## 4. Ask again

Ask the Portland question again, with the new files in place.

In [ ]:
# Ask the agent the lab's question 'm4':
#   "Is the K-600 high-pressure sensor available at the Portland depot?"
!python agent.py --q m4
# Show each tool the newest run called, with its arguments and exactly what it returned.
lab.tool_results()
# A link to the newest run's trace in MLflow.
lab.trace_link()

**This is the module's success signal:** `check_stock` returns `bad_request` naming Denver, Phoenix
and Chicago, and the answer says Portland isn't a depot. In the lab's test runs, it said Portland "is not a
valid depot". The answer must not say the sensor is out of stock or unavailable there.

## 5. Compare with a depot that has no stock (stretch goal)

Skip this step if you're short of time. Two more runs show the other categories: the same sensor at
Denver, a real depot that has none, and module 3's coolant pump question, which the stock data does
cover.

In [ ]:
# Ask the agent this question:
!python agent.py "Is the K-600 high-pressure sensor available at the Denver depot?"
# Ask the agent the lab's question 'm3':
#   "Can we get a replacement coolant pump for a K-400 to the Denver depot this week?"
!python agent.py --q m3
# Show a table of the last 3 runs, one row each: your Portland rerun from step 4, the Denver
# sensor, then the coolant pump.
#   question: the question asked
#   ended by: what stopped the run
#   tools run: the tools the agent ran, and each result's category in parentheses
#   answer: the start of what the agent printed. The full text is in the Answer: lines above
#     the table.
lab.evidence(columns=("question", "ended by", "tools run", "answer"), last=3)

The **tools run** column now shows the category each tool result carried. To see the same in MLflow,
click a run's name in the table to open its trace, then look for a `check_stock` span and open it.
Its outputs should carry the same `category`, `retryable` and `detail`.

**Your read:** after step 5, compare the Denver answer with the first Portland answer from step 1. In
the lab's test runs they're almost word for word the same. Going by the **tools run** column, what
tells you the Denver one is true and the Portland one wasn't?

<details><summary>Check your answer</summary>

The tool result. For Denver, `check_stock` returned `no_results` from a depot that exists: it checked
and found none. For Portland, it returned an empty list that could have meant anything.
</details>

## If the result differs

If Portland's `check_stock` still returns `{"results": []}` with no `category`, the new files aren't
in place. Look at the output of step 3's copy cell: it should list both `tools.py` and `agent.py`. If
it doesn't, or if you ran the reset again after copying, run step 3's copy cell again, then step 4.

If `check_stock` returns `bad_request` but the answer still says the sensor is out of stock or
unavailable at Portland, the success signal isn't met: the tool reported the problem, and the model
didn't pass it on. Check that the `agent.py` comparison in step 3 showed the added lines, then run
step 4 once more. The model's wording can vary between runs. If it still says out of stock, make a
note of it and carry on: module 5 starts from its own reset, with this module's fix in place.

Read the category and the answer together. A confident answer, or a cell that ran without an error,
doesn't show on its own that the agent reported what the tool said.